# 00 - Kafka to Bronze with PySpark Structured Streaming

Optional streaming entry point. Start Kafka with `docker compose up -d kafka`, run `python producer/produce_events.py`, then execute this notebook. Each available Kafka micro-batch is landed through the same local/R2 bronze storage contract.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json

from insightsstream.spark_pipeline import EVENT_SCHEMA, ingest_dataframe_as_bronze
from insightsstream.storage import create_store

spark = (
    SparkSession.builder.master("local[*]")
    .appName("InsightsStream-Kafka-To-Bronze")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)
store = create_store()

In [ ]:
raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "events")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

events = raw.select(from_json(col("value").cast("string"), EVENT_SCHEMA).alias("event")).select("event.*")

def land_micro_batch(batch, batch_id):
    if not batch.isEmpty():
        run = ingest_dataframe_as_bronze(
            batch,
            source_name="kafka://events",
            store=store,
            batch_id=f"kafka-{batch_id:020d}",
        )
        print(f"Landed {run.bronze_rows} Kafka events in {run.bronze_key}")

query = (
    events.writeStream.foreachBatch(land_micro_batch)
    .option("checkpointLocation", str(ROOT / "data" / "checkpoints" / "kafka-to-bronze"))
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()